# Names Analysis

analylize name data from [census.gov](https://www.census.gov/topics/population/genealogy/data/2020_names.html)

### check

In [ ]:
print("kernel is working")

### imports

In [ ]:
from urllib import request
from pathlib import Path
import pandas
import numpy as np
from pandas import DataFrame
from pandas import Series

### Download Data


In [ ]:
ASSETS_DIR = Path("assets")
ASSETS_DIR.mkdir(exist_ok=True)

FIRST_NAMES_URL = r"https://www2.census.gov/topics/genealogy/2020surnames/Names2020_FirstNames_Sex.xlsx"
FIRST_NAMES_FILE = ASSETS_DIR / "first-names.xlsx"
LAST_NAMES_URL = r"https://www2.census.gov/topics/genealogy/2020surnames/Names2020_LastNames_RaceHispanic.xlsx"
LAST_NAMES_FILE = ASSETS_DIR / "last-names.xlsx"

_ = request.urlretrieve(FIRST_NAMES_URL, FIRST_NAMES_FILE)
_ = request.urlretrieve(LAST_NAMES_URL, LAST_NAMES_FILE)

### Load/Visualize Data

In [ ]:
first_names = pandas.read_excel(  # pyright: ignore[reportUnknownMemberType]
    FIRST_NAMES_FILE, skiprows=2
)
first_names = first_names[:-1]  # drop "ALL OTHER NAMES"
first_names

In [ ]:
last_names = pandas.read_excel(  # pyright: ignore[reportUnknownMemberType]
    LAST_NAMES_FILE, skiprows=2
)
last_names = last_names[:-1]  # drop "ALL OTHER NAMES"
last_names = last_names[last_names["LAST NAME"].notna()] # there are some NaN values
last_names

### analysis

In [ ]:
result = first_names.query(  # pyright: ignore[reportUnknownMemberType]
    '`FIRST NAME`=="ALLEN"'
)
result

##### first letter frequencies

In [ ]:
def calculate_letter_frequency(df: DataFrame, column: str) -> Series:
    first_letter = df[column].apply(lambda row: row[0])
    first_names["first_letter"] = first_letter
    first_letters = first_names.groupby("first_letter")
    desired_field = "PROPORTION PER 100,000 POPULATION"
    letter_frequencies = (
        first_letters[desired_field].aggregate(np.sum).sort_values(ascending=False)
    )
    letter_frequencies /= 100000  # into fraction of one
    total_percent = letter_frequencies.sum()
    letter_frequencies /= total_percent  # normalize so sum=1 (currently doesn't because 'other' is droped)
    return letter_frequencies

In [ ]:
letter_frequencies_first_name = calculate_letter_frequency(first_names, "FIRST NAME")
letter_frequencies_first_name

In [ ]:
letter_frequencies_last_name = calculate_letter_frequency(last_names, "LAST NAME")
letter_frequencies_last_name

##### initials

In [ ]:
def get_initials_probability(
    initials: str, first_probabilities: Series, last_probabilities: Series
) -> float:
    first_initial = initials[0]
    last_initial = initials[-1]
    first_match_chance = first_probabilities[first_initial]
    last_match_chance = last_probabilities[last_initial]
    # middle initial
    middle_match_chance = 1
    if len(initials) == 3:
        middle_initial = initials[2]
        middle_match_chance = first_probabilities[middle_initial]
    return first_match_chance * last_match_chance * middle_match_chance


def birthday_problem(match_probability: float, number_of_elements: int) -> float:
    """calculate odds of at least one match in a set of elemenets"""
    miss_probability = 1 - match_probability
    all_miss_probability = miss_probability**number_of_elements
    all_match_probability = 1 - all_miss_probability
    return all_match_probability



In [ ]:

initials = "SB"
match_chance = get_initials_probability(
    initials, letter_frequencies_first_name, letter_frequencies_last_name
)
one_in_x = round(1 / match_chance)
print(f"match chance = {match_chance * 100:.2f} %")
print(f"one in {one_in_x} people")

team_size = 100
match_chance_in_team = birthday_problem(match_chance, team_size)
print(f"match chance in team of {team_size} = {match_chance_in_team*100:.1f} %")
print(f"one in {round(1/match_chance_in_team)} teams")